# MNE-bids

In [ ]:
from pathlib import Path
import numpy as np

import openneuro
import mne
import mne_qt_browser

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)

import matplotlib.pyplot as plt
plt.switch_backend('Qt5Agg')
plt.ion()

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from pop_rejcont import rejcont
from preprocessing import preprocess
from psd_lof import find_bad_channels_psd_lof

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2','med1breath', 'med2']

# Create a BIDS root directory in the MNE dataset folder
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
bids_root.mkdir(parents=True, exist_ok=True)

In [ ]:
subject = "021"
task = 'med2'

In [ ]:
raw: mne.io.edf.edf.RawEDF = read_raw_bids(
    BIDSPath(subject=subject, root=bids_root, task=task, datatype="eeg"),
    verbose=False
)
raw.load_data()
raw.set_montage("colin27_1020", on_missing="warn")
assert type(raw) == mne.io.edf.edf.RawBDF

In [ ]:
preprocess(
    raw,
    reference='EXG6',
    subsample_freq=256,
    high_thresh=None,
    low_thresh=1,
    pop_rejcont=True,
    bad_channel_variance_threshold=None
)

In [ ]:
psd = raw.compute_psd()
psd.plot(picks="eeg", average=False)

In [ ]:
# manually mark bad channels for this ICA exploration
raw.plot(block=True)

In [ ]:
raw.info['bads']

In [ ]:
# now do some ICA stuff
from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

ica = ICA(n_components=20, method='infomax', max_iter="auto", rng=97)
ica.fit(raw, decim = 5)
ica

In [ ]:
ica.plot_components()

In [ ]:
muscle_bads, muscle_bad_scores = ica.find_bads_muscle(raw)
ica.plot_scores(muscle_bad_scores)
ica.plot_properties(raw, picks=muscle_bads)
muscle_bads

In [ ]:
raw.set_channel_types({'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog'})

eog_bads, eog_bad_scores = ica.find_bads_eog(raw)
ica.plot_scores(eog_bad_scores)
ica.plot_properties(raw, picks=eog_bads)
eog_bads

In [ ]:
ica.exclude = muscle_bads + eog_bads

In [ ]:
reconst_raw = raw.copy()
ica.apply(reconst_raw)

In [ ]:
reconst_raw.info

In [ ]:
raw.plot()

In [ ]:
reconst_raw.plot()

In [ ]:
raw.compute_psd().plot()

In [ ]:
reconst_raw.compute_psd().plot()